# Hong Kong Ghost Signs

## Imports, Config, and Constants

In [ ]:
import os
import json

# Datetime manipulation
from datetime import datetime
import datefinder

# Dataframe manipulation
import polars as pl
from polars.exceptions import ComputeError
from polars.datatypes import Datetime, String

# Google Maps and Translate API Clients
import googlemaps
from googletrans import Translator

# Generate UUID
import shortuuid

# Distance Calculation
import haversine as hs
from haversine import Unit


# Setup Google Translator
translator = Translator()

# Configure Polars 
cfg = pl.Config()
cfg.set_tbl_rows(32)
cfg.set_fmt_str_lengths(256)
cfg.set_tbl_width_chars(1000)

# Define Hong Kong Bounding Box
HKBBOX = (22.5639, 114.4013, 22.1771, 113.8373) 

# PATHS to CSVs
DATASRC = "data/hkghostmaps-gmaps-export-20220901.csv"
DATASRC_UUID = "data/hkghostmaps-gmaps-prepared-20220905+uuid.csv"
DATADEST = f"data/hkghostmaps-gmaps-prepared-{datetime.isoformat(datetime.now())[:10]}.csv"

## Load Data
### Generate UUID on first load

In [ ]:
if not os.path.isfile(DATASRC_UUID):
    # Load Features
    df = pl.read_csv(
        source=DATASRC,
        # Prefix columns with 'raw'
        new_columns=[
            'raw_x',
            'raw_y',
            'raw_name',
            'raw_desc',
            'raw_gxml'
        ]).filter(
            # Accept Missing Values
            pl.col("raw_x").eq(0) | (
            # Filter Non-Hong Kong Markers
            (HKBBOX[3] <= pl.col("raw_x")) &
            (pl.col("raw_x") <= HKBBOX[1]) &
            (HKBBOX[2] <= pl.col("raw_y")) &
            (pl.col("raw_y") <= HKBBOX[0]))
        ).sort(by=('raw_x','raw_y'))

    # Add UUID
    def gen_shortuuid(v):
        return shortuuid.uuid()
    
    df = df.with_columns(
        id = pl.col('raw_x').map_elements(gen_shortuuid, return_dtype=str)
    )

    df.write_csv(DATASRC_UUID)
else:
    df= pl.read_csv(DATASRC_UUID)

# Text Parsing

## Parse "raw_name"
### Identify the type of data and language in 'name' field
- Coordinates
- Address
- All Chinese
- Mixed Chinese & English
- All English

In [ ]:
df = df.with_columns(
    rx_name_is_coordinates=pl.col('raw_name').str.contains(r'^(\(22|22)'),
    rx_name_is_address=pl.col('raw_name').str.contains(r' Rd$| St$| Rd W$| Rd E$| Road$| Street$| Strand W$|Rd - Yuen Long$| Ln$'),
    rx_name_is_all_chinese=(
        pl.col('raw_name').str.contains(r'[\u4e00-\u9fff]+')
    ) & ~(pl.col('raw_name').str.contains(r'[a-zA-Z]+')),
    rx_name_is_chinese_and_english=(
        pl.col('raw_name').str.contains(r'[\u4e00-\u9fff]+')
    ) & (pl.col('raw_name').str.contains(r'[a-zA-Z]+'))
)

In [ ]:
# In all other cases, treat the 'name' field to be English
df = df.with_columns(
    rx_name_is_all_english=(
        ~(df['rx_name_is_coordinates']) &
        ~(df['rx_name_is_address']) &
        ~(df['rx_name_is_all_chinese']) &
        ~(df['rx_name_is_chinese_and_english'])
    )
)

## Set English Names from `EN` and mixed `EN/ZH-HANT` 'raw_name' 

#### Set English from pure English

In [ ]:
df = df.with_columns(
    pl.when(pl.col("rx_name_is_all_english"))
    .then(pl.col('raw_name').str.to_titlecase().str.strip_chars())
    .otherwise(None).alias('name'),
    pl.when(pl.col("rx_name_is_all_english"))
    .then(False)
    .otherwise(None).alias('name_gen'),
)

#### Set English from Mixed English and Chinese

In [ ]:
original = df.filter(df['rx_name_is_chinese_and_english'])['raw_name'].to_list()
translations = ["'Ltd. Co.' Under Newer 'Gospel Nursing Home' Sign",
"Illegible Blue Sign on Tile 'Center'",
"Prince 'Prince Edward' Jewelery",
"Sign Behind Air Conditioners Above 'Tai Hung Roast Goose'",
"'Chengdelong' Rice Shop",
"'Zhonghuayang' Illegible",
"HK.Fruit 'Global High-quality Ingredients (Fruits, Seafood, Hairy Crabs)",
"'College' Faded on the 6th Floor.",
"Side of 'Zuiqionglou' Hotel",
"Namkok 'South Point' (Cafe)"]

mixed_to_en = dict(zip(original,translations))

df = df.with_columns(
    pl.when(
        df['rx_name_is_chinese_and_english']
    ).then(
        pl.col('raw_name').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # ~Human translated
    pl.when(
        df['rx_name_is_chinese_and_english']
    ).then(
        False
    ).otherwise(
        pl.col('name_gen')
    ).alias('name_gen'),
)

In [ ]:
# Result, with unprocessed rows
# df.select('name','name_gen').head(10)   

In [ ]:
# Result, only processed rows
# df.filter(df['name_gen'] == False).select('name').head(10)

## Set Traditional Chinese Names from `ZH-HANT` 'raw_name'

In [ ]:
df = df.with_columns(
    pl.when(
        pl.col("rx_name_is_all_chinese")
    ).then(
        pl.col('raw_name').str.strip_chars()
    ).otherwise(
        None
    ).alias('name__zh-hant'),
    pl.when(
        pl.col("rx_name_is_all_chinese")
    ).then(
        False
    ).otherwise(
        None
    ).alias('name__zh-hant_gen'),
)

In [ ]:
# Result, only processed rows
# df.filter(df['name__zh-hant_gen'] == False).select('name__zh-hant').head(10)

## Translate Traditional Chinese Names to English

### Google Translate

In [ ]:
original = [x for x in df['name__zh-hant'].to_list() if x]
translations = [f"'{translator.translate(x, src='zh-tw', dest='en').text}'" for x in original] 

In [ ]:
zhhant_to_en = dict(zip(original,translations))

for src, dest in zhhant_to_en.items():
    print(f'{src} --> {dest}')

In [ ]:
df = df.with_columns(
    pl.when(
        df['rx_name_is_all_chinese']
    ).then(
        pl.col('raw_name').replace_strict(zhhant_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # Machine translated
    pl.when(
        df['rx_name_is_all_chinese']
    ).then(
        True
    ).otherwise(
        pl.col('name_gen')
    ).alias('name_gen'),
)

In [ ]:
# Result, with unprocessed rows
# df.select('name','name_gen','raw_name').head(10)   

## Parse "raw_desc"
### Identify the type of data and language in 'description' field
- Names (where missing `name`)
- Descriptions (where `name` exists too)
- Capture Date of the photo
- Image URL
- All Chinese
- Mixed Chinese & English
- All English

#### `image_url` and `last_seen`

In [ ]:
def extract_date(s):
    try:
        res = list(datefinder.find_dates(s, base_date=datetime(2023,1,1)))[0]
        return res
    except TypeError:
        return None
    except ComputeError:
        return None
    except IndexError:
        return None

pattern_img_elem = r"<img\b[^>]*>"
pattern_desc_img_url=r'src\s*=\s*"([^"]+)"'
pattern_desc_date=r'Captured'
pattern_desc_stripping_img_url_and_date = r"<br><br>(.*?)\s*Captured"
pattern_labeled_date=r"(Captured|Documented|Spotted)\s+\w+\s+\w*\s*(2022|2023|2024)\.?\W?"


# Parse raw name to identify the data entered in the 'description' field 
df = df.with_columns(
    # Image URLs
    rx_img_url_extracted=pl.col('raw_desc').str.extract(pattern_desc_img_url),
    rx_desc_is_img_html=pl.col('raw_desc').str.contains(pattern_desc_img_url),
    # Capture date
    rx_desc_is_captured_date=pl.col('raw_desc').str.contains(pattern_desc_date),
    rx_last_seen_extracted=pl.col('raw_desc').map_elements(extract_date, return_dtype=Datetime),
    # Copy description for multi-step processing
    # ry_desc_before=pl.col('raw_desc').str.replace(pattern_img_elem,"").str.replace_all("<br>",""),
    ry_desc=pl.col('raw_desc').str.replace(pattern_img_elem,"").str.replace_all("<br>","").str.replace(pattern_labeled_date,"").str.strip_chars().replace("", None),
)

In [ ]:
# Result, only Image URL
# df.filter(pl.col('rx_desc_is_img_html'))['rx_img_url_extracted', 'raw_desc']

# Result, only Mixes English / Chinese
# df.filter(pl.col('rx_desc_is_captured_date'))['rx_last_seen_extracted','raw_desc']

In [ ]:
# BEFORE and AFTER inspection
# pl.concat(
#     (df['ry_desc_before'].alias('BEFORE').to_frame(),
#      df['ry_desc'].str.replace(pattern_labeled_date,"").alias('AFTER').to_frame()
#     ),how='horizontal')

In [ ]:
# Manual
manual_desc_encoding = {
    "Corner of Wing Fung & Queen's Road East. Captured while hoarding was up for new development in late April 2023." : "Captured while hoarding was up for new development in late April 2023."
}

df = df.with_columns(
    pl.when(
        pl.col('ry_desc').is_in(list(manual_desc_encoding.keys()))
    ).then(
        pl.col('ry_desc').replace_strict(manual_desc_encoding, default=None)       
    ).otherwise(
        pl.col('ry_desc')
    )
)

#### Classsify Languages

In [ ]:
df = df.with_columns(
    ry_desc_is_all_chinese=(
        pl.col('ry_desc').str.contains(r'[\u4e00-\u9fff]+')
    ) & ~(pl.col('ry_desc').str.contains(r'[a-zA-Z]+')),
    ry_desc_is_chinese_and_english=(
        pl.col('ry_desc').str.contains(r'[\u4e00-\u9fff]+')
    ) & (pl.col('ry_desc').str.contains(r'[a-zA-Z]+')
        )
)

# In all other cases, treat the 'description' field to be English
df = df.with_columns(
    ry_desc_is_all_english=(
        df['ry_desc'].is_not_null() &
        ~(df['ry_desc_is_all_chinese']) &
        ~(df['ry_desc_is_chinese_and_english'])
    )
)

In [ ]:
# Result, All Chinese
# df.filter(pl.col('ry_desc_is_all_chinese'))['ry_desc']
# Result, Mixed English / Chinese
# df.filter(pl.col('ry_desc_is_chinese_and_english'))['ry_desc']
# Result, All English
# df.filter(pl.col('ry_desc_is_all_english'))['ry_desc']

## Determine which All English descriptions can be promoted to the English name

In [ ]:
desc_to_name_exceptions = [
    "Revealed sign after a car accident broke off a chunk of the building.",
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory.",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete."
]

desc_to_name_exception_names = {
    "Revealed sign after a car accident broke off a chunk of the building." : "Revealed By Car Accident",
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory." : "Ghost Sign Trio",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete." : "Dragon Phoenix Photography Studio",
    "'同' above Healthy Wood on the side next to garage" : "Together"
}

desc_to_name_exception_descs = {
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory." : "Trading Company, Plumber and Electrician, Garment Factory",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete." : "Along with an unreadable sign remnant on concrete",
    "(one day before it was scheduled to be painted over)" : "The day before it was scheduled to be painted over",
    "寶 something trading - vacant shop with for lease sign over the shop front" : "Vacant shop with for lease sign over the shop front",
    "金堂大廈 sign under air conditioning units over 3 shop" : "Under air conditioning units over 3 shop",
    '億和公司 revealed after vegetable stall closed'  : "Revealed after vegetable stall closed",
    '地產代理 in white in side alley.': "In white, found in a side alley",
    '灸 next to Stern Rockwell.' : "Next to Stern Rockwell",
    'Duplicate jeans sign of the one being covered down the road Po Sang Trading 寶生易公司' : "Duplicate jeans sign of the one being covered down the road Po Sang Trading",
    '中國肉公司 - behind roll up gate.' : "Behind a roll up gate",
    '陳董仁 painted over in yellow on Reclamation Street' : "Painted over in yellow on Reclamation Street",
    "'智' on corner of Nanking and Shanghai Street" : "On corner of Nanking and Shanghai Street",
    "Chinese Medicine and '牛王' red letters on blue tile" : "Red letters on blue tile",
    "'同' above Healthy Wood on the side next to garage" : "Above Healthy Wood on the side next to garage"
}

df = df.with_columns(
    pl.when(
        (pl.col('name').is_null()) &
        (pl.col('ry_desc_is_all_english')) &
        (pl.col('name').is_null()) &
        ~(pl.col('ry_desc').is_in(desc_to_name_exceptions))
    ).then(
        pl.col('ry_desc').str.to_titlecase().str.strip_chars().str.replace(r'$\(','').str.replace(r'\)^','')
    ).otherwise(
        pl.col('name')
    ).alias(
        'name'
    ),
    # Set Name Generated Field to False
    pl.when(
        (pl.col('name').is_null()) &
        (pl.col('ry_desc_is_all_english')) &
        (pl.col('name').is_null()) &
        ~(pl.col('ry_desc').is_in(desc_to_name_exceptions))
    ).then(
        False
    ).otherwise(
        pl.col("name_gen")
    ).alias('name_gen'),
    # Set Description Field to None -- silly code
    pl.when(
        True      
    ).then(
        None
    ).otherwise(
        None
    ).alias('description'),   
    # Set Description Generated Field to False
    pl.when(
        True
    ).then(
        None
    ).otherwise(
        None
    ).alias('description_gen')   
)

# Manual Correction of Names
df = df.with_columns(
    pl.when(
        pl.col('ry_desc').is_in(list(desc_to_name_exception_names.keys()))
    ).then(
        pl.col('ry_desc').replace_strict(desc_to_name_exception_names, default=None)  
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    pl.when(
        pl.col('ry_desc').is_in(list(desc_to_name_exception_names.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('name_gen')
    ).alias('name_gen'),
)

# Manual Correction of Descriptions
df = df.with_columns(
    pl.when(
        pl.col('ry_desc').is_in(list(desc_to_name_exception_descs.keys()))
    ).then(
        pl.col('ry_desc').replace_strict(desc_to_name_exception_descs, default=None)    
    ).otherwise(
        pl.col('description')
    ).alias('description'),
    pl.when(
        pl.col('ry_desc').is_in(list(desc_to_name_exception_descs.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('description_gen')
    ).alias('description_gen'),
)


## Determine which Mixed English / Chinese descriptions can be promoted to the English name

In [ ]:
original_still_missing_name = df.filter((df['name'].is_null()) & (df['ry_desc_is_chinese_and_english']))['ry_desc'].to_list()

translations = [
    "'Precious' Something Trading",
    "'Jintang Building'",
    "'Yihe Company'",
    "'Estate agency'",
    "'Moxibustion'",
    "Behind 'Cheng Hing' and 'Defa Tea Restaurant'",
    "Multiple 'Runfa' Signs",
    "'Jeans'",
    "Illegible 'Company'",
    "Side of building 'Country _ Fixed Head'",
    "'Big Welding Supply Co.'",
    "'Yajiu Engineering Co. Ltd.'",
    "'China Meat Company'",
    "'Chen Dongren'",
    "'Wisdom'",
     "'West Branch', then Illegible",
    "Chinese Medicine and 'Ox King'",
   "'Together'",
 ]

mixed_to_en = dict(zip(original_still_missing_name,translations))

df = df.with_columns(
    pl.when(
       (df['name'].is_null()) & (df['ry_desc_is_chinese_and_english'])
    ).then(
        pl.col('ry_desc').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # ~Human translated
    pl.when(
       (df['name'].is_null()) & (df['ry_desc_is_chinese_and_english'])
    ).then(
        False
    ).otherwise(
        pl.col('name_gen')
    ).alias('name_gen'),
)

In [ ]:
# Results, processed rows
# df.filter((df['name'].is_not_null()) & (df['ry_desc_is_chinese_and_english']))['name','description', 'ry_desc']

## Determine which Mixed English / Chinese descriptions can be translated and used as the English description

In [ ]:
original_already_have_names = df.filter(
        (df['name_gen'] != False) & 
        (df['name'].is_not_null()) & 
        (df['ry_desc_is_chinese_and_english'])
    )['ry_desc'].to_list()

translations = [
 "Above dessert place",
 "In the back alley behind Ping Pong",
 "In yellow",
 "Illegible 'hardware' - behind car repair shop sign",
 "Double layer with paint peeling off to reveal 'Lu Apartment Shanghai'",
 "'Lin_Wen` Chinese Doctor and 2 or 3 more illegible to the right",
 "'Irobu' painted over sign - off Jordan Road",
 "'Peace' under signage from old Standard Chartered Branch"
]

mixed_to_en = dict(zip(original_already_have_names,translations))

df = df.with_columns(
    pl.when(
        (df['name_gen'] != False) & (df['name'].is_not_null()) & (df['ry_desc_is_chinese_and_english'])
    ).then(
        pl.col('ry_desc').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('description')
    ).alias('description'),
    # ~Human translated
    pl.when(
       (df['name_gen'] != False) & (df['name'].is_not_null()) & (df['ry_desc_is_chinese_and_english'])
    ).then(
        False
    ).otherwise(
        pl.col('description_gen')
    ).alias('description_gen'),
)

In [ ]:
# Results, processed rows
# df.filter((df['description'].is_not_null()) & (df['ry_desc_is_chinese_and_english']))['id','name','description','ry_desc']

In [ ]:
# Manual correction / cleanup
name_for_id = {
    "JSrHippq8sn6487Soy57Uj" : "'Fuhua Tea Shop'",
    "mWFQstFZ7EHYZoa4UoF4vb" : "'Ming Hua Noodle House'",
    "PrhRzZGd3Q3WPUqTZzAaPD" : "'He Kee Hardware'",
    "GRARP8Q9YhUMZpybV4bHwX" : "'Pan Ji Wood Paint'"
}

df = df.with_columns(
    pl.when(
        df['id'].is_in(list(name_for_id.keys()))
    ).then(
        pl.col('id').replace_strict(name_for_id, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name')
)

## Set Traditional Chinese names from `ZH-HANT` 'raw_desc'

In [ ]:
df = df.with_columns(
    pl.when(
        (pl.col("ry_desc_is_all_chinese")) &
        (pl.col('name__zh-hant').is_null())
    ).then(
        pl.col('ry_desc').str.strip_chars().str.strip_chars('.')
    ).otherwise(
        pl.col('name__zh-hant')
    ).alias('name__zh-hant'),
    
    pl.when(
        (pl.col("ry_desc_is_all_chinese")) &
        (pl.col('name__zh-hant').is_null())
    ).then(
        False
    ).otherwise(
        pl.col('name__zh-hant_gen')
    ).alias('name__zh-hant_gen')
)

In [ ]:
# Result, only processed rows
# df.filter((pl.col("ry_desc_is_all_chinese")) & (df['name__zh-hant_gen'] == False)).select('name__zh-hant')

## Determine which Mixed English / Chinese descriptions can be used as the Traditional Chinese name

In [ ]:
mapping_id_to_name_zh_hant = {
    "MoUenBfFNz8kwiC9cYv7xE" : "'寶__'",
    "SXzbKw562zzZMGxrMeD2b2" : "'金堂大廈'",
    "h6H8bHYcNCMRNu6JQZtZas" : "'億和公司'",
    "nzAhQWc66HPtsif9nfffeU" : "'成德隆'",
    "b5xgzdeidUsg7edwKBCBx9" : "'地產代理'",
    "5CGpNw9S78bvcXHCB5TyXv" : "'灸'",
    "aw7wLFXS6gYrP9sv63Sfok" : "'成興和德發茶餐廳'",
    "mCEPQKUwFCy8cYZkjuRbBf" : "'潤發行'",
    "jueyvJ3o9mVo7QbQpEDFCV" : "'寶生易公司'",
    "J3ZgttCaPZV8STErD5Rec4" : "難以辨認'公司'",
    "fFLpYi6t9TL3VG62JRzaDg" : "'國_定頭'",
    "U7PDZpzB9HCYAaP2tM6BAw" : "'中華洋'難以辨認",
    "TjSqbaRW8REbwjiQWkqyR7" : "'大溶接器材行'",
    "bA3Xh4sivm59KjQMow5j8w" : "'雅就工程有限公司'",
    "7dFfj8vVzee6ZaTHBLCEVg" : "'陳董仁'",
    "4QEUmAmDHbXxohLBbewewT" : "'智'",
    "gmyBUNrK6gARB9qP5gm3E8" : "'西可'",
    "AXPkkwLhy7SoMYAFEs7KLu" : "'牛王'",
    "oGoev2mBpKKCu82CYNqGKQ" : "'學院'",
    "JuxA3HQde3Gi4WdrsEyEsN" : "'醉瓊楼飯店'",
    "axrHTxU4wm8KHPCCKFdu9F" : "'尖尖照相'",
    "h2tw4jFNCFLMZENhV6btKf" : "'龍'不完整的",
    "Eu5gBHGa7c4AWo4f7fp7Lg" : "'同'"
}

df = df.with_columns(
    pl.when(
        df['id'].is_in(list(mapping_id_to_name_zh_hant.keys()))
    ).then(
        pl.col('id').replace_strict(mapping_id_to_name_zh_hant, default=None)
    ).otherwise(
        pl.col('name__zh-hant')
    ).alias('name__zh-hant'),
    
    pl.when(
        df['id'].is_in(list(mapping_id_to_name_zh_hant.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('name__zh-hant_gen')
    ).alias('name__zh-hant_gen')
)

In [ ]:
# Result, processed rows
# df.filter(
#    (df['name__zh-hant'].is_not_null()) &  
#    (df['ry_desc_is_chinese_and_english'])
# )['id','name__zh-hant','name__zh-hant_gen','ry_desc']

## Promote Traditional Chinese descriptions from `raw_desc` to description, if they differ from the Traditional Chinese Name

In [ ]:
df = df.with_columns(
    pl.when(
        pl.col('rx_name_is_all_chinese') &
        pl.col('ry_desc_is_all_chinese') &
        ((pl.col('name__zh-hant')) != (pl.col('ry_desc').str.strip_chars('.')))
    ).then(
        pl.col('ry_desc')
    ).otherwise(
        None
    ).alias('description__zh-hant'),
    # NOT Machine translated
    pl.when(
        pl.col('rx_name_is_all_chinese') &
        pl.col('ry_desc_is_all_chinese') &
        ((pl.col('name__zh-hant')) != (pl.col('ry_desc').str.strip_chars('.')))
    ).then(
        False
    ).otherwise(
        None
    ).alias('description__zh-hant_gen'),
)

In [ ]:
# Result, 
# df[
#  'name',
#  'name_gen',
#  'name__zh-hant',
#  'name__zh-hant_gen',
#  'description',
#  'description_gen',
# 'description__zh-hant',
# 'description__zh-hant_gen',
#  'raw_name',
#  'ry_desc',
# 'rx_name_is_all_chinese',
#  'ry_desc_is_all_chinese'
# ].filter(
#     pl.col('rx_name_is_all_chinese') &
#     pl.col('ry_desc_is_all_chinese') &
#     ((pl.col('name__zh-hant')) != (pl.col('ry_desc').str.strip_chars('.')))
# )


## Translate English names from `ZH-HANT` 'raw_desc' 

### Google Translate

In [ ]:
original_still_missing_name = df.filter(
    (df['name'].is_null()) & 
    (df['ry_desc_is_all_chinese'])
)['ry_desc'].to_list()

translations = [f"'{translator.translate(x, src='zh-tw', dest='en').text}'" for x in original_still_missing_name]

zhhant_to_en = dict(zip(original_still_missing_name,translations))

for src, dest in zhhant_to_en.items():
    print(f'{src} --> {dest}')

In [ ]:
df = df.with_columns(
    pl.when(
        (df['ry_desc_is_all_chinese'])&
        (df['name']).is_null()
    ).then(
        pl.col('ry_desc').replace_strict(zhhant_to_en, default=None).str.to_titlecase()
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # Machine translated
    pl.when(
        df['ry_desc_is_all_chinese']
    ).then(
        True
    ).otherwise(
        pl.col('name_gen')
    ).alias('name_gen'),
)

In [ ]:
# Result, only processed rows
# df.filter(
#     (df['ry_desc_is_all_chinese']) &
#     (df['name_gen'])
# ).select('name','name_gen','ry_desc')

## Translate English descriptions from `ZH-HANT` 'raw_desc' 

In [ ]:
original_already_have_names_but_missing_desc = df.filter(
        (df['name'].is_not_null()) &
        (df['name_gen'] != True) & 
        (df['description'].is_null()) & 
        (df['ry_desc_is_all_chinese'])
    )['ry_desc'].to_list()

Since this is a null set, no further processing is required.

## Set `last_seen` date

In [ ]:
df = df.cast({pl.Datetime: pl.Date})
df = df.with_columns(
    last_seen=pl.col('rx_last_seen_extracted')
)

## Set `image_url` 

In [ ]:
df = df.with_columns(
    image_url=pl.col('rx_img_url_extracted')
)

## Review Parsing Results

In [ ]:
col_order = [
'id',

'name',
'name_gen',
'name__zh-hant',
'name__zh-hant_gen',

'description',
'description_gen',
'description__zh-hant',
'description__zh-hant_gen',

'raw_name',
'ry_desc',

'last_seen',
'image_url',

'rx_last_seen_extracted',
'rx_img_url_extracted',

'rx_name_is_all_english',
'rx_name_is_all_chinese',
'rx_name_is_chinese_and_english',
'rx_name_is_coordinates',
'rx_name_is_address',

'ry_desc_is_all_english',
'ry_desc_is_all_chinese',
'ry_desc_is_chinese_and_english',
'rx_desc_is_img_html',
'rx_desc_is_captured_date',

'raw_desc',
'raw_x',
'raw_y',
'raw_gxml'
]

df = df.select(col_order)

In [ ]:
# df[
#     'name',
#     'name_gen',
#     'name__zh-hant',
#     'name__zh-hant_gen',
#     'description',
#     'description_gen',
#     'description__zh-hant',
#     'description__zh-hant_gen',
#     'raw_name',
#     'ry_desc',
#     'rx_name_is_all_chinese',
#     'ry_desc_is_all_chinese'
# ].filter(
#     # pl.col('name').is_null()
#     True
# )

# Geocoding

## Fill missing Latitude & Longitude coordinates

Because the Dataframe is sorted by Latitude, the rows with missing values show up first with a `0` value. There are five of them.

In [ ]:
# Addresses for GhostSigns with missing coordinates
df['raw_name'].to_list()[:5]

In [ ]:
original = df.filter(df['raw_x'] == 0)['raw_name'].to_list()
encodings = [
    (22.328725033185375, 114.18915132846354),
    (22.330166285778667, 114.19210082344293),
    (22.32950210461055, 114.19222392266883),
    (22.32935639108943, 114.1922340848458),
    (22.324942897750564, 114.18932768694137)
]

lats = [lat for (lat,lng) in encodings]
lngs = [lng for (lat,lng) in encodings]

address_to_lat = dict(zip(original,lats))
address_to_lng = dict(zip(original,lngs))

df = df.with_columns(
    pl.when(
        df['raw_x'] == 0
    ).then(
        pl.col('raw_name').replace_strict(address_to_lng, default=None)
    ).otherwise(
        pl.col('raw_x')
    ).alias('longitude'),
    pl.when(
        df['raw_y'] == 0
    ).then(
        pl.col('raw_name').replace_strict(address_to_lat, default=None)
    ).otherwise(
        pl.col('raw_y')
    ).alias('latitude'), 
)

In [ ]:
# Result, with unprocessed rows
# df.select('name','longitude','latitude').head(10)   

## Reverse Geocoding - Get Address from Coordinates

### Setup Google Maps client with our API Key

In [ ]:
env_vars = {}
with open('../.env') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        key, value = line.strip().split('=', 1)
        env_vars[key] = value

gmaps = googlemaps.Client(env_vars['PUBLIC_GOOGLEMAPS_KEY'])

### Define how the Google Maps API response will be used in our data schema

In [ ]:
geoDB = {}

# Mapping from Google API address component name, to our component names
component_prefix = {
    'administrative_area_level_1' : 'administrativeAreaLevel1',
    'country' : "country",
    'neighborhood': "neighbourhood",
    'premise' : "premise",
    'route' : 'route',
    'street_number': 'streetNumber',
    'plus_code' : "plusCode",
    'subpremise' : 'subPremise',
    'intersection' : 'intersection'
}

# Template for Address properties (Formatted Address and Components in EN, ZH-HANT and ZH-HANS, and Metadata)
AddressProperties = {
  "distanceFromPoint" : None,
  "formattedAddress" : None,
  "formattedAddress__zh-hant" : None,
  "formattedAddress__zh-hans" : None,
  "plusCode" : None,
  "plusCode__zh-hant" : None,
  "plusCode__zh-hans" : None,
  "subPremise" : None,
  "subPremise__zh-hant" : None,
  "subPremise__zh-hans" : None,
  "premise" : None,
  "premise__zh-hant" : None,
  "premise__zh-hans" : None,
  "streetNumber" : None,
  "streetNumber__zh-hant" : None,
  "streetNumber__zh-hans" : None,
  "route" : None,
  "route__zh-hant" : None,
  "route__zh-hans" : None,
  "neighbourhood" : None,
  "neighbourhood__zh-hant" : None,
  "neighbourhood__zh-hans" : None,
  "administrativeAreaLevel1" : None,
  "administrativeAreaLevel1__zh-hant" : None,
  "administrativeAreaLevel1__zh-hans" : None,
  "country" : None,
  "country__zh-hant" : None,
  "country__zh-hans" : None,
  "googlePlaceId" : None,
  "geocoder" : None,
  "reverseGen" : False,
  "forwardGen" : False,
} 

def parse_to_address_properties(id, result, src_latlng:tuple, lang="en", geocoder='googlemaps', reverse_encode=True):
    lang_suffix = {
        "en": "",
        "zh-hk": "__zh-hant",
        "zh-cn": "__zh-hans",
    }.get(lang.lower())


    address_properties = AddressProperties.copy() if id not in geoDB else geoDB[id]

    if not len(result) > 0:
        raise IndexError
    
    result = result[0]
    
    for comp in result['address_components']:
        for t in comp['types']:
            if t in ['political','transit_station','bus_station','tourist_attraction', 'point_of_interest', 'establishment', 'park']:
                continue
            field = f'{component_prefix[t]}{lang_suffix}'
            address_properties[field] = comp['long_name']
            
    field = f'formattedAddress{lang_suffix}'
    address_properties[field] = result["formatted_address"]
    
    field = 'distanceFromPoint'
    latlng = result["geometry"]["location"]
    address_properties[field] = int(hs.haversine(
        src_latlng,
        (latlng['lat'], latlng['lng']),
        unit=Unit.METERS
    ))
    
    field = "googlePlaceId"
    address_properties[field] = result["place_id"]

    field = "geocoder"
    address_properties[field] = geocoder

    field = "reverseGen"
    address_properties[field] = reverse_encode

    field = "forwardGen"
    address_properties[field] = reverse_encode == False

    address_properties['id'] = id
    
    geoDB[id] = address_properties

### Request Google Maps API for each LAT,LNG coordinate, and parse the result to Address Properties

We cache the API response so we can use that on subsequent runs

In [ ]:
for row in df.select('id','raw_name','latitude','longitude').rows(named=True):
    print(f"GEOFEATURE :: {row['id']} :: {row['raw_name']}")
    
    # Look up an address with reverse geocoding
    for lang in ['en','zh-HK','zh-CN']:
        JSONPATH = f'data/{row['id']}-{lang}-gmaps-reverse-geocoded.json'
        if not os.path.isfile(JSONPATH):
            reverse_geocode_result = gmaps.reverse_geocode(
                (row['latitude'], row['longitude']),
                result_type=[
                    'subpremise',
                    'premise',
                    'establishment',
                    'street_number',
                    'street_address',
                    'point_of_interest',
                    'parking',
                    'colloquial_area',
                    'sublocality',
                    'plus_code'

                ],
                location_type=['ROOFTOP','RANGE_INTERPOLATED','GEOMETRIC_CENTER'],
                language=lang
            )

            with open(JSONPATH, 'w', encoding='utf-8') as f:
                json.dump(reverse_geocode_result, f, ensure_ascii=False, indent=4)
        else:
            # Open and read the JSON file
            with open(JSONPATH, 'r') as file:
                reverse_geocode_result = json.load(file)
        
        parse_to_address_properties(row['id'], reverse_geocode_result, (row['latitude'], row['longitude']), lang=lang)

In [ ]:
# Example result
# geoDB['Czi8oDy7NuZwAtrLhPFkXP']

### Merge all Address Properties with Dataframe

In [ ]:
df_geocoded = pl.from_dicts(list(geoDB.values()))
df = df.join(df_geocoded, on="id")

In [ ]:
# Example result
# df.filter(pl.col('id')=='Czi8oDy7NuZwAtrLhPFkXP')

## Forward Geocoding - Get Address Components from Address

The idea is that the address that was originally captured in the dataset is likely to be more accurate than the reverse geocoded address obtained from the coordinates, so we will overwrite the Address Properties of those records where an address was provided.

### Request Google Maps API for each 'raw_name' that was an address, and parse the result to Address Properties

We cache the API response so we can use that on subsequent runs

In [ ]:
geoDB = {}

for row in df.select('id','raw_name','rx_name_is_address', 'latitude', 'longitude').rows(named=True):
    
    # Only encode raw names which were also addresses
    if not row['rx_name_is_address']:
        continue
        
    print(f"GEOFEATURE :: {row['id']} :: {row['raw_name']}")
    
    # Look up an address with forward geocoding
    for lang in ['en', 'zh-HK', 'zh-CN']:
        JSONPATH = f'data/{row['id']}-{lang}-gmaps-forward-geocoded.json'
        
        if not os.path.isfile(JSONPATH):
            print('> Google API?')
            geocode_result = gmaps.geocode(
                row['raw_name'],
                components={"country":"HK"},
                language=lang
            )

            with open(JSONPATH, 'w', encoding='utf-8') as f:
                json.dump(geocode_result, f, ensure_ascii=False, indent=4)
        else:
            # Open and read the JSON file
            with open(JSONPATH, 'r') as file:
                geocode_result = json.load(file)
                
        parse_to_address_properties(row['id'], geocode_result, (row['latitude'], row['longitude']), lang=lang, reverse_encode=False)

### Define the merge logic for how the new Address Properties should overwrite the columns in the Dataframe

In [ ]:
# The new address components have an intersection component, so we need to backfill this into our original Dataframe
df = df.with_columns(
    pl.lit(None).alias('intersection'),
    pl.lit(None).alias('intersection__zh-hant'),
    pl.lit(None).alias('intersection__zh-hans'),
)

In [ ]:
def update(self:pl.DataFrame, df_other:pl.DataFrame,  join_columns:list[str]) -> pl.DataFrame:
    '''Updates dataframe with the values in df_other after joining on the join_columns'''
    
    # The columns that will be updated
    columns = [c for c in df_other.columns if c not in join_columns]
    
    df_merged = (self
        .join(df_other, how='left', on=join_columns, suffix='_NEW')      
        .with_columns(**{
            c: pl.coalesce([pl.col(c+'_NEW'), pl.col(c)]) for c in columns # <-This updates the columns
        }).select(
            pl.all().exclude('^.*_NEW$') # <- this drops the temporary '*_NEW' columns
           )
       )    
    return df_merged

### Merge the Forward Encoded Address Properties with Dataframe

In [ ]:
df_forward_encoded = pl.from_dicts(list(geoDB.values()))
# df.join(df_forward_encoded, on="id", how='left', coalesce=True)
df = update(df, df_forward_encoded, join_columns=['id'])

# Exporting

## Review Result

In [ ]:
col_order = [
'id',

'name',
'name_gen',
'name__zh-hant',
'name__zh-hant_gen',

'description',
'description_gen',
'description__zh-hant',
'description__zh-hant_gen',

'raw_name',
'ry_desc',

'longitude',
'latitude',
'distanceFromPoint',
    
'formattedAddress',
'formattedAddress__zh-hant',
'formattedAddress__zh-hans',

'last_seen',
'image_url',

'plusCode',
'plusCode__zh-hant',
'plusCode__zh-hans',
'subPremise',
'subPremise__zh-hant',
'subPremise__zh-hans',
'premise',
'premise__zh-hant',
'premise__zh-hans',
'streetNumber',
'streetNumber__zh-hant',
'streetNumber__zh-hans',
'intersection',
'intersection__zh-hant',
'intersection__zh-hans',
'route',
'route__zh-hant',
'route__zh-hans',
'neighbourhood',
'neighbourhood__zh-hant',
'neighbourhood__zh-hans',
'administrativeAreaLevel1',
'administrativeAreaLevel1__zh-hant',
'administrativeAreaLevel1__zh-hans',
'country',
'country__zh-hant',
'country__zh-hans',

'googlePlaceId',
'geocoder',
'reverseGen',
'forwardGen',

'rx_last_seen_extracted',
'rx_img_url_extracted',

'rx_name_is_all_english',
'rx_name_is_all_chinese',
'rx_name_is_chinese_and_english',
'rx_name_is_coordinates',
'rx_name_is_address',

'ry_desc_is_all_english',
'ry_desc_is_all_chinese',
'ry_desc_is_chinese_and_english',
'rx_desc_is_img_html',
'rx_desc_is_captured_date',

'raw_desc',
'raw_x',
'raw_y',
'raw_gxml'
]

df = df.select(col_order)

In [ ]:
# cfg.set_tbl_rows(10)
# df.select('name','name__zh-hant','longitude','latitude', 'formattedAddress','formattedAddress__zh-hant','formattedAddress__zh-hans')

## Save Result

In [ ]:
df.write_csv(DATADEST)

## Temporary Interpolated Results

In [ ]:
# DESC : Central Pier 5
# DESC : DHL Express Service Point
# DESC : New Treasure Centre

In [ ]:
# "'Strong Hardware'",
#  "'Remove'",
#  "'Songji Daxin Fashion'",
 
#  "'WenJi decoration project' in yellow",
#  "'Ming Hua Noodle House'",
#  "'Hongchang Tea House'",
#  "'He Kee Hardware'",
#  "'Shengli'",
#  "'Xinglu Shiduo'",
#  "'Pan Ji Wood Paint'",
#  "'Telemo Pump'",
#  "'Fourth floor of 'Hanglebie'",
#  "'Oshidahexie'",
#  "'Ya Nguyen'",
#  "'Huang Ti Nong's Western Clothes'",